# Paper figures

Each figure is built from cached per-model artifacts under `$ANALYSIS_DIR/<model>/`.

- **Models** are defined once in `conf/analysis/models.yaml`; the `MODELS` list below picks which ones (and in what order) to compare.
- **Extraction** happens in SLURM jobs, not here. Run the *preflight* cell to see what is missing and submit `extract.sbatch` jobs; come back once they finish.
- Each figure cell returns a `Figure` shown inline and saved to `report/figures/<name>.pdf`.

Kernel: the project `.venv`.

In [1]:
import sys, pathlib
REPO = pathlib.Path.cwd()
while not (REPO / 'pyproject.toml').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))

import matplotlib
matplotlib.use('Agg')  # non-interactive: figures only render when explicitly displayed

from IPython.display import display
from protein_design.wandb_plots import set_publication_style
from protein_design.analysis import figures, registry
set_publication_style()

# --- the models to compare (keys from conf/analysis/models.yaml) ---
# Add / remove / reorder freely; order is the plotting order.
MODELS = [
    'vanilla_35m',
    'evo_35m',
    'evo_c05_blosum25',
    'vanilla_650m',
    'evo_650m',
    'just_dpo_4ep_step1376',
]

# For the embedding probe (DPO-starting-base question), compare the 650M family:
# vanilla -> OAS-evo -> C05 cdrmix (ppl-best) -> C05 cdrmix (Spearman-best).
# The last two are the same run; if best-Spearman scores higher on the probe than
# best.pt, that is the drift signature (early checkpoint = better DPO init).
PROBE_MODELS = [
    'vanilla_650m',
    'evo_650m',
    'evo_c05_cdrmix',
    'evo_c05_cdrmix_spearman',
]

# Datasets to use (a list of keys, or 'all' for every dataset in the registry).
DATASETS = 'all'

print('known models:', sorted(registry.load_models()))
print('known datasets:', registry.dataset_keys('all'))

known models: ['c05_blosum25_seed', 'evo_35m', 'evo_650m', 'evo_c05_blosum25', 'evo_c05_cdrmix', 'evo_c05_cdrmix_spearman', 'evo_dedup_35m', 'just_dpo_4ep_step1376', 'vanilla_35m', 'vanilla_650m']
known datasets: ['ed2_m22', 'ed5_m22', 'ed811_m22', 'ed2_si06', 'ed5_si06', 'exp']


## Preflight — what still needs extracting

Prints the exact `sbatch` commands for any missing artifacts. Run them in a terminal (or submit from here with `subprocess`), then re-run the figure cells once the jobs finish.

In [2]:
missing = figures.preflight(MODELS, DATASETS, kind='pll')
# Copy the sbatch commands above into a terminal to submit the extraction jobs.
# Return here and re-run the figure cells once the jobs finish.

All pll artifacts present for 6 model(s) x 6 dataset(s).


## Figure: PLL vs experimental enrichment (Spearman)

Grouped bars — one bar per model per dataset. Higher = the model's CDR-H3 pseudo-log-likelihood ranks experimental binding enrichment better.

In [ ]:
fig = figures.plot_pll_spearman(MODELS, DATASETS)
out = figures.save_fig(fig, 'pll_spearman')  # prints: Saved: <path>
display(fig)

Saved: /cluster/home/mdenegri/protein-design/report/figures/pll_spearman.pdf


<Figure size 1230x630 with 1 Axes>

## Preflight — scorer predictions

Scorer predictions (M22/SI06 binder scorer) are computed once per dataset with `score_dms.sbatch` (uses the `flu` conda env). They don't vary per model — run this only if the scorer CSV is missing.

In [5]:
missing_scorer = figures.preflight_scorer(DATASETS)
# Uncomment to submit:
# import subprocess
# for d in missing_scorer:
#     subprocess.run(["sbatch", "bash_scripts/utils/score_dms.sbatch", "--dataset", d], check=True)

All scorer artifacts present.


## Figure: PLL vs supervised scorer (Spearman)

Grouped bars show Spearman(PLL, truth) per model per dataset — same as the figure above. The **dashed horizontal segment** per dataset is Spearman(scorer, truth): the fixed binder scorer's correlation with experimental truth. This is the supervised upper bound; bars above the dashed line mean the model's PLL is a better ranker than the scorer.

In [6]:
fig = figures.plot_pll_vs_scorer_spearman(MODELS, DATASETS)
figures.save_fig(fig, 'pll_vs_scorer_spearman')
fig

Saved: /cluster/home/mdenegri/protein-design/report/figures/pll_vs_scorer_spearman.pdf


<Figure size 1230x630 with 1 Axes>

## Figure: CDR-H3 pseudo-perplexity

Per-sequence pseudo-perplexity PP = exp(−PLL / L), where L is the CDR-H3 length. **Lower PP = model assigns higher average probability to each residue.** Sequences are pooled across the selected datasets (deduplicated by sequence string).

In [7]:
fig = figures.plot_pseudo_perplexity(MODELS, DATASETS)
figures.save_fig(fig, 'pseudo_perplexity')
fig

Saved: /cluster/home/mdenegri/protein-design/report/figures/pseudo_perplexity.pdf


<Figure size 1080x600 with 1 Axes>

## Embedding probe — is the binding signal linearly decodable?

A frozen-backbone **linear probe** measures how well each model's CDR-H3 representation
already organizes the DMS sequences by binding: standardize, fit Ridge + kNN on the DMS
**train** split, score Spearman/R² on the **test** split. Higher = the embedding space
encodes binding more accessibly → a better starting base for DPO (especially small-data).
The PCA figure below is illustrative; the probe table is the actual evidence.

Embeddings are extracted in SLURM jobs (`--what emb`) and cached under
`$ANALYSIS_DIR/<model>/emb/<dataset>.npz`; re-running is a no-op.

**Run the probe on a compute node, not here.** The `emb` artifacts are up to ~2 GB each
(343k × 1280) and the probe fits over the full train split — this OOMs the login node. A
SLURM job does the heavy lifting and writes the results to `report/figures/`:

```
sbatch bash_scripts/probe.sbatch          # -> probe_scores.csv, probe_spearman.pdf, embedding_pca.pdf
```

The cells below just read those results back (cheap, login-node-safe).

In [2]:
# Preflight: which embedding artifacts still need extracting (copy cmds into a terminal).
missing_emb = figures.preflight_emb(PROBE_MODELS, DATASETS)
# import subprocess
# for m in sorted({m for m, _ in missing_emb}):
#     ds = ",".join(d for mm, d in missing_emb if mm == m)
#     subprocess.run(["sbatch", "bash_scripts/extract.sbatch",
#                     "--what", "emb", "--model", m, "--dataset", ds], check=True)

Missing 6 emb artifact(s). To extract:

  sbatch bash_scripts/extract.sbatch --what emb --model evo_650m --dataset ed2_m22,ed5_m22,ed811_m22,ed2_si06,ed5_si06,exp


In [3]:
# Probe comparison table — read the results computed by `sbatch bash_scripts/probe.sbatch`.
# (Don't call figures.probe_scores() here: it loads ~2 GB npz per model and OOMs the login node.)
import pandas as pd
csv_path = figures.FIGURES_DIR / 'probe_scores.csv'
if not csv_path.exists():
    raise FileNotFoundError(f"{csv_path} not found — run: sbatch bash_scripts/probe.sbatch")
probe_df = pd.read_csv(csv_path)
display(probe_df.round(3))

,model,label,dataset,ridge_spearman,ridge_r2,knn_spearman,n_train,n_test,eval
0,vanilla_650m,ESM2-650M,ed2_m22,0.530,0.439,0.542,220283,27536,train->test
1,vanilla_650m,ESM2-650M,ed5_m22,0.761,0.707,0.752,343056,42884,train->test
2,vanilla_650m,ESM2-650M,ed811_m22,0.673,0.617,0.608,282967,35430,train->test
3,vanilla_650m,ESM2-650M,ed2_si06,0.595,0.507,0.593,73012,9011,train->test
4,vanilla_650m,ESM2-650M,ed5_si06,0.733,0.767,0.732,181472,22578,train->test
5,vanilla_650m,ESM2-650M,exp,0.642,0.587,0.622,102956,102956,5-fold cv
6,evo_c05_cdrmix,C05 cdrmix (best.pt),ed2_m22,0.524,0.439,0.550,220283,27536,train->test
7,evo_c05_cdrmix,C05 cdrmix (best.pt),ed5_m22,0.758,0.696,0.755,343056,42884,train->test
8,evo_c05_cdrmix,C05 cdrmix (best.pt),ed811_m22,0.669,0.604,0.628,282967,35430,train->test
9,evo_c05_cdrmix,C05 cdrmix (best.pt),ed2_si06,0.588,0.500,0.607,73012,9011,train->test


In [4]:
# Bar chart from the cached table (cheap — just plots probe_df, no embeddings loaded).
# The job already saved probe_spearman.pdf; this re-renders it inline. Pass
# metric='knn_spearman' to show the kNN probe instead of ridge.
fig = figures.plot_probe_spearman(PROBE_MODELS, DATASETS, df=probe_df)
figures.save_fig(fig, 'probe_spearman')
display(fig)

Saved: /cluster/home/mdenegri/protein-design/report/figures/probe_spearman.pdf


<Figure size 1230x630 with 1 Axes>

In [5]:
# PCA scatter — generated by the SLURM job (needs the full embeddings, too heavy for here).
# Display the saved PDF rather than recomputing.
from IPython.display import IFrame
pca_pdf = figures.FIGURES_DIR / 'embedding_pca.pdf'
if not pca_pdf.exists():
    raise FileNotFoundError(f"{pca_pdf} not found — run: sbatch bash_scripts/probe.sbatch")
IFrame(str(pca_pdf.relative_to(REPO / 'report')), width='100%', height=400)

## Probe learning curve — the small-data regime (DPO starting base)

The bars above saturate: with 200k–340k labelled train examples a linear probe extracts
the binding signal regardless of how the backbone was tuned, so representation differences
wash out (all models ≈ equal). The regime that actually matters for a **DPO starting base**
is *small-data*: how few labels each model needs. The learning curve subsamples the train
split to increasing sizes (fixed test, averaged over seeds) — a model whose line sits
**higher at small n_train** organizes binding more accessibly.

Computed by the SLURM job (`--mode curve`); the cells below read the saved results.

```
sbatch bash_scripts/probe.sbatch --mode curve     # -> learning_curve.csv, learning_curve.pdf
```

In [6]:
# Learning curve from the cached results (cheap — reads learning_curve.csv, no embeddings).
lc_path = figures.FIGURES_DIR / 'learning_curve.csv'
if not lc_path.exists():
    raise FileNotFoundError(f"{lc_path} not found — run: sbatch bash_scripts/probe.sbatch --mode curve")
lc_df = pd.read_csv(lc_path)
fig = figures.plot_learning_curve(PROBE_MODELS, DATASETS, df=lc_df)
figures.save_fig(fig, 'learning_curve')
display(fig)

Saved: /cluster/home/mdenegri/protein-design/report/figures/learning_curve.pdf


<Figure size 1890x1020 with 6 Axes>